# 02 Descriptive table by cohort (Table S2)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import utils

In [ ]:
cohort_sheets = {
    "São Paulo 2012–2014 (training)": "train_2012_2014",
    "São Paulo 2015 (validation)": "validation_2015",
    "São Paulo 2016 local (MoH)": "test_local_2016_moh",
    "Brazil 2016 (external)": "test_external_brazil_2016",
    "São Paulo 2016 (CEInfo)": "test_ceinfo_2016",
    "São Paulo 2017–2020 (temporal)": "test_temporal_sp_2017_2020",
}
cohorts = {sheet: utils.load_object(utils.DATA + "/" + name) for sheet, name in cohort_sheets.items()}

# the external sheet describes the municipalities other than São Paulo city
brazil = cohorts["Brazil 2016 (external)"]
cohorts["Brazil 2016 (external)"] = brazil[brazil["CODMUNNASC"] != utils.SP_CODE]
{sheet: cohort.shape for sheet, cohort in cohorts.items()}

In [ ]:
# row order and category decoding
variable_order = [
    ("outcome", "OBITO", "Neonatal mortality"),
    ("continuous", "IDADEMAE", "Mother's age"),
    ("categorical", "ESCMAE2010", "Mother's education", "Fairness"),
    ("categorical", "RACACORMAE", "Race and ethnicity", "Fairness"),
    ("categorical", "PARTO", "Mode of delivery", "Predictor"),
    ("categorical", "LOCNASC", "Place of delivery", "Predictor"),
    ("continuous", "SEMAGESTAC", "Gestational age"),
    ("continuous", "PESO", "Birth weight"),
    ("categorical", "IDANOMAL", "Congenital anomaly", "Predictor"),
    ("continuous", "APGAR1", "1st Apgar"),
    ("continuous", "APGAR5", "5th Apgar"),
]
category_labels = {
    "ESCMAE2010": {0: "0 - No education", 1: "1 - Elementary", 2: "2 - Middle",
                   3: "3 - High school", 4: "4 - Incomplete superior", 5: "5 - Complete superior"},
    "RACACORMAE": {1: "1 - White", 2: "2 - Black", 3: "3 - Yellow", 4: "4 - Brown", 5: "5 - Indigenous"},
    "PARTO": {1: "1 - Vaginal", 2: "2 - Cesarean"},
    "LOCNASC": {1: "1 - Hospital", 2: "2 - Other health facility", 3: "3 - Residence"},
    "IDANOMAL": {1: "1 - Yes", 2: "2 - No"},
}
variable_order

In [ ]:
# continuous as median (IQR), categorical as count (percentage) per category; one sheet per cohort
sheets_s2 = {}
for sheet_name, cohort in cohorts.items():
    survived = cohort[cohort.OBITO == 0]
    died = cohort[cohort.OBITO == 1]
    live_col = "Live births (n=" + format(len(cohort), ",") + ")"
    survived_col = "Neonatal survived (n=" + format(len(survived), ",") + ")"
    death_col = "Neonatal death (n=" + format(len(died), ",") + ")"
    rows = []
    for item in variable_order:
        kind, column, name = item[0], item[1], item[2]
        if kind == "outcome":
            rows.append({"Type": "Outcome", "Characteristics": name,
                         live_col: "", "Missing (%)": "", survived_col: "", death_col: ""})
        elif kind == "continuous":
            rows.append({"Type": "Predictor", "Characteristics": name,
                         live_col: utils.median_iqr_text(cohort[column]),
                         "Missing (%)": utils.missing_percent_text(cohort[column]),
                         survived_col: utils.median_iqr_text(survived[column]),
                         death_col: utils.median_iqr_text(died[column])})
        else:
            rows.append({"Type": item[3], "Characteristics": name, live_col: "",
                         "Missing (%)": utils.missing_percent_text(cohort[column]),
                         survived_col: "", death_col: ""})
            for code_value, label in category_labels[column].items():
                rows.append({"Type": "", "Characteristics": label,
                             live_col: utils.count_percent_text(cohort[column], code_value),
                             "Missing (%)": "",
                             survived_col: utils.count_percent_text(survived[column], code_value),
                             death_col: utils.count_percent_text(died[column], code_value)})
    sheets_s2[sheet_name] = pd.DataFrame(rows, columns=["Type", "Characteristics", live_col,
                                                        "Missing (%)", survived_col, death_col])
for sheet, table in sheets_s2.items():
    print(sheet)
    display(table)

In [ ]:
utils.save_sheets(sheets_s2, utils.SUPPLEMENT_TABLES, "Table_S2")